In [26]:
#Installs the libraries for language models, text embeddings, PDF reading, data handling, and model acceleration.
!pip install -q transformers sentence-transformers pypdf pandas accelerate

In [2]:
#Imports the tools used later, sets the data folder and model names, and checks whether a GPU is available.
import os, json, torch
import numpy as np
import pandas as pd
from transformers import pipeline
from sentence_transformers import SentenceTransformer

DATA_DIR = "/content/sample_data/mars-ai-chatbot"
LLM_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
EMBED_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [11]:
#Shows the files in the data folder and their sizes so you can see what the chatbot can use.
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        p = os.path.join(root, f)
        print(p, f"{os.path.getsize(p)/1024:.0f} KB")

/content/sample_data/mars-ai-chatbot/mars_qa_train.jsonl 17 KB
/content/sample_data/mars-ai-chatbot/README.md 3 KB
/content/sample_data/mars-ai-chatbot/mars_knowledge.jsonl 15 KB
/content/sample_data/mars-ai-chatbot/mars_knowledge_supplement.jsonl 30 KB
/content/sample_data/mars-ai-chatbot/44695_Planet-Mars-Quick-Facts_Mars-nasa-gov.pdf 338 KB
/content/sample_data/mars-ai-chatbot/mars_qa_validation.jsonl 4 KB
/content/sample_data/mars-ai-chatbot/mars_qa_test.jsonl 4 KB
/content/sample_data/mars-ai-chatbot/dataset_quality_report.md 5 KB
/content/sample_data/mars-ai-chatbot/mars_sources.csv 4 KB


In [12]:
#Defines helpers to read text, PDF, CSV, and JSON files, convert their contents into text, and split longer text into smaller chunks.
def decode(b):
    try: return b.decode("utf-8")
    except UnicodeDecodeError: return b.decode("cp1252", errors="replace")

def read_text(path):
    with open(path, "rb") as f:
        return "\n".join(decode(l) for l in f.read().splitlines())

def chunk_text(text, size=800, overlap=100):
    text = " ".join(text.split())
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size].strip())
        start += size - overlap
    return [c for c in chunks if len(c) > 30]

def flat(v):
    if isinstance(v, dict):
        return "; ".join(f"{k}: {flat(x)}" for k, x in v.items() if flat(x))
    if isinstance(v, (list, tuple)):
        return ", ".join(flat(x) for x in v if flat(x))
    if v is None or (isinstance(v, float) and v != v):
        return ""
    return str(v).strip()

def record_to_text(rec):
    if isinstance(rec, dict):
        msgs = rec.get("messages")
        if isinstance(msgs, list) and msgs and all(isinstance(m, dict) and "role" in m for m in msgs):
            q = " ".join(m.get("content", "") for m in msgs if m["role"] == "user")
            a = " ".join(m.get("content", "") for m in msgs if m["role"] == "assistant")
            return f"Q: {q}\nA: {a}"
        return "\n".join(f"{k}: {flat(v)}" for k, v in rec.items() if flat(v))
    return flat(rec)

def read_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        text = ""
        try:
            from pypdf import PdfReader
            text = " ".join((p.extract_text() or "") for p in PdfReader(path).pages)
        except Exception:
            pass
        if len(text.strip()) < 50:                 # fallback extractor
            try:
                import fitz
                with fitz.open(path) as doc:
                    text = " ".join(p.get_text() for p in doc)
            except Exception:
                pass
        if len(text.strip()) < 50:
            print("   (no text found: probably an image-only PDF, needs OCR)")
        return chunk_text(text)
    if ext == ".csv":
        df = pd.read_csv(path, on_bad_lines="skip", encoding_errors="replace")
        return [record_to_text(r) for r in df.to_dict("records")]
    if ext == ".jsonl":
        out = []
        for line in read_text(path).splitlines():
            if line.strip():
                try: out.append(record_to_text(json.loads(line)))
                except json.JSONDecodeError: pass
        return out
    if ext == ".json":
        data = json.loads(read_text(path))
        rows = data if isinstance(data, list) else [data]
        return [record_to_text(r) for r in rows]
    try:                                            # any other file: try as plain text
        return chunk_text(read_text(path))
    except Exception:
        return []

In [15]:
#Creates a text file with information about India’s Mars Orbiter Mission so the chatbot can retrieve it later.
india = """India's Mars Orbiter Mission (MOM), also called Mangalyaan, was ISRO's first interplanetary mission.
It launched on 5 November 2013 from Sriharikota on a PSLV rocket and entered orbit around Mars on 24 September 2014.
India became the first Asian country to reach Mars orbit, and the first country in the world to do so on its first attempt.
The mission cost roughly 450 crore rupees (about 74 million US dollars). It was designed to last six months but kept
operating for several years, until contact with the spacecraft was lost in 2022.
Its five instruments were the Mars Colour Camera, Thermal Infrared Imaging Spectrometer, Methane Sensor for Mars,
Lyman Alpha Photometer, and Mars Exospheric Neutral Composition Analyser.
India reached Mars orbit with an orbiter. It has not landed a spacecraft on the surface of Mars."""
with open(f"{DATA_DIR}/india_mars_mission.txt", "w") as f:
    f.write(india)


In [16]:
# Meta-files aren't Mars facts; test/validation are kept out so you can evaluate fairly later
#Reads supported files, skips the listed metadata and test files, and collects their text chunks and source filenames.
EXCLUDE = {"README.md", "dataset_quality_report.md", "mars_qa_test.jsonl", "mars_qa_validation.jsonl"}

chunks, sources = [], []
for root, _, files in os.walk(DATA_DIR):
    for name in sorted(files):
        if name in EXCLUDE:
            continue
        path = os.path.join(root, name)
        try:
            got = read_file(path)
        except Exception as e:
            print(f"SKIPPED {name}: {e}")
            continue
        chunks += got
        sources += [name] * len(got)
        print(f"{name}: {len(got)} chunks")

print(f"\nTotal: {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0] if chunks else "NO DATA LOADED")

   (no text found: probably an image-only PDF, needs OCR)
44695_Planet-Mars-Quick-Facts_Mars-nasa-gov.pdf: 0 chunks
india_mars_mission.txt: 2 chunks
mars_knowledge.jsonl: 26 chunks
mars_knowledge_supplement.jsonl: 66 chunks
mars_qa_train.jsonl: 32 chunks
mars_sources.csv: 13 chunks

Total: 139 chunks

Sample chunk:
 India's Mars Orbiter Mission (MOM), also called Mangalyaan, was ISRO's first interplanetary mission. It launched on 5 November 2013 from Sriharikota on a PSLV rocket and entered orbit around Mars on 24 September 2014. India became the first Asian country to reach Mars orbit, and the first country in the world to do so on its first attempt. The mission cost roughly 450 crore rupees (about 74 million US dollars). It was designed to last six months but kept operating for several years, until contact with the spacecraft was lost in 2022. Its five instruments were the Mars Colour Camera, Thermal Infrared Imaging Spectrometer, Methane Sensor for Mars, Lyman Alpha Photometer, and 

In [27]:
#Converts each text chunk into an embedding (a numerical representation of its meaning). The retrieve function finds the chunks most similar to a question.
embedder = SentenceTransformer(EMBED_NAME, device="cuda" if torch.cuda.is_available() else "cpu")
chunk_vecs = embedder.encode(chunks, normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print("Index shape:", chunk_vecs.shape)

def retrieve(question, k=3):
    q = embedder.encode(question, normalize_embeddings=True)
    scores = chunk_vecs @ q
    top = np.argsort(-scores)[:k]
    return [(chunks[i], sources[i], float(scores[i])) for i in top]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Index shape: (139, 384)


In [7]:
#Loads the text-generation model that will write the chatbot’s replies, using the GPU if one is available.
mars_bot = pipeline(
    'text-generation',
    model=LLM_NAME,
    device=0 if torch.cuda.is_available() else -1,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
print("Model loaded.")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Model loaded.


In [28]:
#Searches for relevant text, adds it to the question when a good match is found, and asks the language model to generate a reply.
SIM_THRESHOLD = 0.35

SYSTEM = ("You are a helpful assistant who specializes in Mars and space exploration, "
          "but you can answer general questions too. If reference context is provided, "
          "use it and prefer it over your own memory. If no context is provided, answer "
          "from your own knowledge. If you are not sure about something, say so instead "
          "of guessing. Never mention 'the context' or 'the document' in your answer.")

def ask_mars_assistant(user_question):
    hits = [h for h in retrieve(user_question, k=3) if h[2] >= SIM_THRESHOLD]

    if hits:
        context = "\n\n".join(h[0] for h in hits)
        user_content = f"Context:\n{context}\n\nQuestion: {user_question}"
    else:
        user_content = user_question

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_content}
    ]

    prompt = mars_bot.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    response = mars_bot(
        prompt,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.4,
        pad_token_id=mars_bot.tokenizer.eos_token_id
    )
    return response[0]['generated_text'][len(prompt):].strip()

In [24]:
#Starts a text chat in the notebook. It keeps accepting questions until you type exit or quit.
print("Mars Bot: Hello! Ask me anything about Mars and Mars exploration.")
print("(Type 'exit' or 'quit' to stop)\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ['exit', 'quit']:
        print("Mars Bot: Goodbye!")
        break
    if not user_input.strip():
        continue

    try:
        print(f"Mars Bot: {ask_mars_assistant(user_input)}\n")
    except Exception as e:
        print(f"Mars Bot: I ran into an error processing that: {e}\n")

Mars Bot: Hello! Ask me anything about Mars and Mars exploration.
(Type 'exit' or 'quit' to stop)

You: exit
Mars Bot: Goodbye!


In [25]:
#Builds a styled Mars-themed chat page with example questions, a send button, and a clear-chat button. It connects the page to the answer function and launches a shareable web link.
import gradio as gr

# ---------------------------------------------------------------
#  Look & feel (all CSS lives inside gr.HTML so it works on any Gradio version)
# ---------------------------------------------------------------
STYLE = """
<style>
  body, .gradio-container { background:#05060f !important; }
  .gradio-container { max-width:1100px !important; margin:auto !important; }
  footer { display:none !important; }

  .hero { position:relative; overflow:hidden; text-align:center; color:#f6e9e2;
          padding:56px 16px 40px; border-radius:24px;
          background:radial-gradient(ellipse at 50% 0%, #2a0f0a 0%, #0a0612 55%, #05060f 100%); }
  .stars, .stars2 { position:absolute; inset:0; pointer-events:none; }
  .stars  { background-image:
      radial-gradient(1px 1px at 20px 30px,#fff,transparent),
      radial-gradient(1.5px 1.5px at 90px 140px,#fff,transparent),
      radial-gradient(1px 1px at 160px 70px,#ffd9c7,transparent),
      radial-gradient(2px 2px at 220px 190px,#fff,transparent),
      radial-gradient(1px 1px at 300px 40px,#fff,transparent);
      background-size:320px 240px; animation:drift 140s linear infinite; opacity:.9; }
  .stars2 { background-image:
      radial-gradient(1px 1px at 60px 90px,#cfe3ff,transparent),
      radial-gradient(1.5px 1.5px at 180px 20px,#fff,transparent),
      radial-gradient(1px 1px at 250px 160px,#fff,transparent);
      background-size:410px 300px; animation:drift 220s linear infinite reverse; opacity:.7; }
  @keyframes drift { to { background-position:-640px 480px; } }

  .tag { position:relative; letter-spacing:.35em; font-size:.8rem; color:#ff9a6b; text-transform:uppercase; }
  .title { position:relative; margin:6px 0 4px; font-size:clamp(3.2rem,12vw,7rem); font-weight:900;
           letter-spacing:.18em; line-height:1;
           background:linear-gradient(180deg,#ffd2b8 0%,#ff7a45 45%,#b3380c 100%);
           -webkit-background-clip:text; background-clip:text; color:transparent;
           filter:drop-shadow(0 0 28px rgba(255,100,50,.45)); animation:rise 1.2s ease-out both; }
  .sub { position:relative; max-width:640px; margin:0 auto 30px; color:#d9c4ba; font-size:1.05rem;
         animation:rise 1.6s ease-out both; }
  @keyframes rise { from { opacity:0; transform:translateY(24px); } to { opacity:1; transform:none; } }

  .planet-wrap { --s:min(300px,68vw); position:relative; width:var(--s); height:var(--s);
                 margin:10px auto 34px; animation:float 7s ease-in-out infinite; }
  @keyframes float { 50% { transform:translateY(-10px); } }
  .tex { position:absolute; inset:0; border-radius:50%; background-color:#c4501f;
         background-image:
           radial-gradient(ellipse 14% 10% at 20% 55%, rgba(80,25,12,.65), transparent 70%),
           radial-gradient(ellipse 22% 12% at 58% 36%, rgba(110,40,20,.55), transparent 70%),
           radial-gradient(ellipse 12% 9% at 44% 74%, rgba(70,20,10,.6), transparent 70%),
           radial-gradient(ellipse 26% 14% at 80% 62%, rgba(150,60,30,.55), transparent 70%),
           radial-gradient(ellipse 18% 8% at 35% 25%, rgba(235,140,90,.45), transparent 70%);
         background-size:calc(var(--s)*2) 100%; animation:spin 38s linear infinite; }
  @keyframes spin { to { background-position-x:calc(var(--s)*2); } }
  .shade { position:absolute; inset:0; border-radius:50%;
           background:radial-gradient(ellipse 30% 9% at 50% 4%, rgba(255,255,255,.75), transparent 75%);
           box-shadow:inset -48px -34px 90px rgba(0,0,0,.82), inset 10px 10px 40px rgba(255,205,175,.16),
                      0 0 90px rgba(235,100,50,.38), 0 0 200px rgba(235,100,50,.15); }
  .orbit { position:absolute; inset:-34px; animation:orbit 15s linear infinite; }
  .orbit i { position:absolute; top:0; left:50%; width:14px; height:14px; border-radius:50%;
             background:radial-gradient(circle at 35% 35%,#d8cfc7,#6f655d); box-shadow:0 0 10px rgba(255,255,255,.35); }
  .orbit.two { inset:-70px; animation-duration:26s; animation-direction:reverse; }
  .orbit.two i { width:9px; height:9px; }
  @keyframes orbit { to { transform:rotate(360deg); } }

  .stats { position:relative; display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr));
           gap:12px; max-width:880px; margin:0 auto 30px; }
  .stat { padding:16px 10px; border-radius:16px; background:rgba(255,255,255,.05);
          border:1px solid rgba(255,140,90,.22); backdrop-filter:blur(6px);
          transition:transform .25s, border-color .25s, box-shadow .25s; }
  .stat:hover { transform:translateY(-5px); border-color:rgba(255,140,90,.7); box-shadow:0 8px 30px rgba(255,100,50,.2); }
  .stat b { display:block; font-size:1.55rem; color:#ffb089; }
  .stat span { font-size:.78rem; letter-spacing:.08em; text-transform:uppercase; color:#b9a59b; }

  .cta { position:relative; display:inline-block; padding:14px 34px; border-radius:999px; text-decoration:none;
         font-weight:700; letter-spacing:.06em; color:#fff !important;
         background:linear-gradient(135deg,#ff7a45,#c1440e); box-shadow:0 0 30px rgba(255,100,50,.5);
         animation:pulse 2.4s ease-in-out infinite; }
  @keyframes pulse { 50% { box-shadow:0 0 52px rgba(255,100,50,.85); transform:scale(1.04); } }

  .chat-head { text-align:center; margin:34px 0 12px; color:#f6e9e2; }
  .chat-head h2 { margin:0; font-size:1.7rem; letter-spacing:.08em; color:#ff9a6b; }
  .chat-head p  { margin:4px 0 0; color:#b9a59b; }

  #mars-chat { background:rgba(14,8,18,.75) !important; border:1px solid rgba(255,120,70,.3) !important;
               border-radius:18px !important; box-shadow:0 0 40px rgba(255,90,40,.12); }
  #mars-input textarea, #mars-input input { background:#12080e !important; color:#fff !important;
               border:1px solid rgba(255,120,70,.35) !important; border-radius:12px !important; }
  #send-btn { background:linear-gradient(135deg,#ff7a45,#c1440e) !important; color:#fff !important; border:0 !important; font-weight:700 !important; }
  .chip button, .chip { background:rgba(255,255,255,.05) !important; color:#f0d9cd !important;
               border:1px solid rgba(255,140,90,.3) !important; border-radius:999px !important; }
  .chip:hover { border-color:#ff7a45 !important; }
</style>
"""

HERO = """
<div class="hero">
  <div class="stars"></div><div class="stars2"></div>
  <div class="tag">Mission Control · Online</div>
  <div class="title">MARS</div>
  <div class="sub">The Red Planet: home to the tallest volcano in the solar system, canyons that dwarf the Grand Canyon,
  and the best chance we have of finding life beyond Earth.</div>

  <div class="planet-wrap">
    <div class="orbit"><i></i></div>
    <div class="orbit two"><i></i></div>
    <div class="tex"></div><div class="shade"></div>
  </div>

  <div class="stats">
    <div class="stat"><b>24h 39m</b><span>Length of a day</span></div>
    <div class="stat"><b>687 days</b><span>Length of a year</span></div>
    <div class="stat"><b>38%</b><span>Earth's gravity</span></div>
    <div class="stat"><b>-63 °C</b><span>Average temperature</span></div>
    <div class="stat"><b>2</b><span>Moons: Phobos &amp; Deimos</span></div>
    <div class="stat"><b>~22 km</b><span>Height of Olympus Mons</span></div>
  </div>

  <a class="cta" href="#chat-anchor">Ask Mission Control ↓</a>
</div>
<div id="chat-anchor" class="chat-head">
  <h2>🔴 MARS MISSION CONTROL</h2>
  <p>Ask anything. Answers draw on your NASA documents first.</p>
</div>
"""

# ---------------------------------------------------------------
#  Chat logic (works with old and new Gradio chat formats)
# ---------------------------------------------------------------
try:
    chatbot = gr.Chatbot(type="messages", height=460, show_label=False, elem_id="mars-chat")
    USE_MESSAGES = True
except TypeError:   # newer Gradio: messages format is the only one
    chatbot = gr.Chatbot(height=460, show_label=False, elem_id="mars-chat")
    USE_MESSAGES = int(gr.__version__.split(".")[0]) >= 6

def chat_fn(message, history):
    history = history or []
    if not message or not message.strip():
        return "", history
    try:
        reply = ask_mars_assistant(message)
    except Exception as e:
        reply = f"Signal lost 📡 ({e})"
    if USE_MESSAGES:
        history = history + [{"role": "user", "content": message},
                             {"role": "assistant", "content": reply}]
    else:
        history = history + [(message, reply)]
    return "", history

EXAMPLES = [
    "What is Mars?",
    "How long is a day on Mars?",
    "Why is Mars red?",
    "What did the Perseverance rover discover?",
    "Tell me about India's Mars mission",
]

# ---------------------------------------------------------------
#  Layout
# ---------------------------------------------------------------
with gr.Blocks(title="Mars Mission Control") as demo:
    gr.HTML(STYLE + HERO)
    chatbot.render()
    with gr.Row():
        msg = gr.Textbox(placeholder="Ask about Mars, rovers, missions...", show_label=False,
                         container=False, scale=8, elem_id="mars-input")
        send = gr.Button("Launch 🚀", scale=1, elem_id="send-btn")
    with gr.Row():
        for ex in EXAMPLES:
            b = gr.Button(ex, size="sm", elem_classes="chip")
            b.click(lambda x=ex: x, None, msg).then(chat_fn, [msg, chatbot], [msg, chatbot])
    clear = gr.Button("Clear chat", size="sm")

    msg.submit(chat_fn, [msg, chatbot], [msg, chatbot])
    send.click(chat_fn, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: ("", []), None, [msg, chatbot])

demo.queue().launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://184086b1f91c7d9d44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7861 <> https://184086b1f91c7d9d44.gradio.live
